> ## ⚠️ متجاوَز — شغّل الخادم محلياً بدل هذا
>
> **هذا الدفتر يشتغل، بس ما عاد لازم.** انكتب لمّا كان المفترض إن جهاز ٨ غيغا رام و٤ غيغا
> VRAM ما يتحمّل ١٤ نموذج سوى. **القياس رفض الافتراض** (2026-09-06، نفس الجهاز):
>
> | القياس | القيمة |
> |---|---|
> | زمن الإقلاع | < ٤٥ ثانية |
> | النماذج المتاحة | **١٤/١٤** |
> | الذاكرة المقيمة | ١.٢٥ غيغا (٣.٣٧ ملتزَمة) |
> | `brain` على صورة حقيقية | **0.13 – 0.69 ثانية** |
> | `chest` | **0.50 ثانية** |
> | `symptoms_ar` (عربي) | **0.76 ثانية** |
>
> توقّعنا بطءاً لأن ثلثي العملية بملف الصفحات. ما صار: النموذج المستخدَم يبقى مقيم،
> فالاستدلال دون الثانية.
>
> **الطريق الموصى بيه:** دبل كليك على `start_server.bat`، وافتح `Radiology Hub.html`.
> ماكو حزمة، ولا Drive، ولا نفق، ولا رابط عام.
>
> **متى يرجع ينفع هذا الدفتر؟** لمّا تحتاج توصل الخادم **من جهاز ثاني** (موبايل، أو تعرضه
> لأحد)، أو إذا الجهاز المحلي انشغل بتدريب. ساعتها شغّل `api/make_serving_bundle.py`
> واتبع الخلايا تحت.

---

# تشغيل خادم AI Radiology Hub على Colab

هذا الدفتر يشغّل `api/main.py` **كما هو** على بطاقة Colab، ويفتح نفق ngrok حتى توصله من أي جهاز.
ما يدرّب ولا نموذج، وما يغيّر ولا رقم — بس يخدم الي موجود.

**ليش أصلاً؟** الجهاز المحلي ٨ غيغا رام و٤ غيغا VRAM، وتحميل كل النماذج سوى يخنقه.
كولاب عنده ~١٢ غيغا رام و١٦ غيغا VRAM على T4 — نفس الكود، تنفّس أوسع.

**ما يحتاج مفتاح Anthropic.** يحتاج شغلتين بس: توكن ngrok، وحزمة الأوزان من Drive.

---

### ⚠️ اقرا هذي قبل ما تشغّل

| الحد | التفصيل |
|---|---|
| **الرابط عام** | أي واحد عنده رابط ngrok يوصل الـAPI ويرفع صور. `main.py:41` مضبوط `allow_origins=["*"]`. لا تنشر الرابط، وأطفي النفق لمّا تخلص (آخر خلية). |
| **الجلسة مؤقتة** | كولاب يقطع بعد ~٩٠ دقيقة خمول و~١٢ ساعة كحد أقصى. هذا خادم عرض وتجربة، **مو استضافة**. |
| **الرابط يتبدّل** | ngrok المجاني يعطي عنواناً جديداً كل تشغيل. |
| **صفحة ngrok الاعتراضية** | الحساب المجاني يعرض صفحة «Visit Site» أول مرة. اضغطها مرة وحدة — بعدها الكوكي يخلّي طلبات `fetch` تمر، لأن الصفحة والـAPI بنفس الأصل. |
| **ليس جهازاً طبياً** | نفس تحفّظات المشروع تنطبق: أداة تعليمية. |

## ١) فحص البيئة

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU    : %s | %.1f GB" % (p.name, p.total_memory / 1e9))
else:
    # لا assert هنا: الخدمة تشتغل على المعالج بلا مشاكل، بس الطلب الواحد أبطأ.
    # التدريب هو الي يستوجب بطاقة، مو الخدمة.
    print("GPU    : ماكو — راح يشتغل على المعالج (يشتغل، بس أبطأ لكل طلب)")

## ٢) توكن ngrok

الخلية تجرّب عدة أسماء شائعة للسر، وتفرّق بين حالتين كولاب يخلطهن على الناس:

* `SecretNotFoundError` — السر **ما موجود** أصلاً.
* `NotebookAccessError` — السر موجود بس **مفتاح «Notebook access» مطفي** لهذا الدفتر.

خذ التوكن من `dashboard.ngrok.com` ← *Your Authtoken*.

In [ ]:
from google.colab import userdata
try:
    from google.colab.userdata import SecretNotFoundError, NotebookAccessError
except ImportError:      # أسماء الاستثناءات تبدّلت بين إصدارات كولاب
    SecretNotFoundError = NotebookAccessError = Exception

CANDIDATES = ["NGROK_AUTH_TOKEN", "NGROK_AUTHTOKEN", "NGROK_TOKEN", "NGROK_API_KEY"]

NGROK_TOKEN, denied = None, []
for name in CANDIDATES:
    try:
        v = userdata.get(name)
        if v and v.strip():
            NGROK_TOKEN = v.strip()
            print("[ok] استخدمت السر:", name)
            break
    except NotebookAccessError:
        denied.append(name)          # موجود، بس بدون إذن — هذي أهم معلومة نعطيها
    except Exception:
        pass                          # غير موجود، أو اسم ثاني

if NGROK_TOKEN is None:
    if denied:
        print("[!] هذي الأسرار موجودة بس هذا الدفتر ما عنده إذن عليها: %s" % ", ".join(denied))
        print("    افتح 🔑 Secrets وشغّل مفتاح «Notebook access» للسر — وجوده وحده ما يكفي.")
    else:
        print("[!] ما لكيت سر بأي اسم من: %s" % ", ".join(CANDIDATES))
        print("    🔑 Secrets ← Add new secret ← الاسم NGROK_AUTH_TOKEN ← القيمة من dashboard.ngrok.com")
        print("    ← وشغّل «Notebook access».")
    print("\n    أو الصقه هنا مؤقتاً (ينمسح بانتهاء الجلسة):")
    print('    NGROK_TOKEN = "2abc..."')

## ٣) حزمة الأوزان

الأوزان **مو بالجت** (`.gitignore` يستثني `*.pt`)، فلازم توصل بطريق ثاني.

**على جهازك، مرة وحدة:**

```bash
cd api
venv/Scripts/python.exe make_serving_bundle.py
```

يطلع `radiology_serving_bundle.zip` (~١.٤ غيغا) فيه **بالضبط** الي يحتاجه الخادم: الأوزان
المخدومة، ومصادر `api/*.py`، وصفحة الواجهة. ارفعه لـ**Google Drive** مرة وحدة، وبعدها كل
جلسة كولاب تسحبه من هناك.

> السكربت يترك ٢٨٪ من `models/` وراه عن قصد: نسخ v1 الي `main.py` ما يفتحها لأن v2 موجودة،
> وبذور الـensemble المتصادمة، و`_ar_v1_backup`. كلها ملفات الخادم ما يلمسها.

In [ ]:
import glob, os, time, zipfile
from google.colab import drive

drive.mount("/content/drive")

HUB = "/content/hub"
PATTERNS = ["/content/drive/MyDrive/**/radiology_serving_bundle.zip",
            "/content/drive/Shareddrives/**/radiology_serving_bundle.zip",
            "/content/**/radiology_serving_bundle.zip"]

BUNDLE = None
for pat in PATTERNS:
    hits = sorted(glob.glob(pat, recursive=True))
    if hits:
        BUNDLE = hits[0]
        break

if BUNDLE is None:
    raise SystemExit(
        "ما لكيت radiology_serving_bundle.zip بـDrive.\n"
        "شغّل api/make_serving_bundle.py على جهازك وارفع الـzip لـMyDrive، ثم أعِد هذي الخلية.")

print("bundle :", BUNDLE, "(%.2f GB)" % (os.path.getsize(BUNDLE) / 1e9))

# نفك لقرص كولاب المحلي مو نقرا من Drive مباشرة: Drive عبر FUSE بطيء وينقطع أحياناً،
# والخادم يفتح ١.٤ غيغا أوزان وقت الإقلاع — انقطاع واحد بالنص يعني نموذج ناقص بصمت.
os.makedirs(HUB, exist_ok=True)
t0 = time.time()
with zipfile.ZipFile(BUNDLE) as z:
    names = z.namelist()
    for i, n in enumerate(names, 1):
        z.extract(n, HUB)
        if i % 20 == 0 or i == len(names):
            print("  فككت %d/%d" % (i, len(names)), end="\r", flush=True)
print("\nفك الضغط بـ %.0f ثانية" % (time.time() - t0))

API_DIR = os.path.join(HUB, "api")
n_pt = len(glob.glob(os.path.join(API_DIR, "models", "*.pt")))
print("api/   :", API_DIR, "| %d checkpoint" % n_pt)
print("الصفحة :", "موجودة" if os.path.exists(os.path.join(HUB, "Radiology Hub.html")) else "غايبة (الـAPI راح يرد JSON بس)")

## ٤) المكتبات

`--no-deps` مقصودة لـ`torchxrayvision` و`medmnist`: الاثنان يعلنان اعتماداً على torch،
وتركهما يحلّان اعتمادهما بحرية يخلي pip **يعيد تنصيب torch** ويكسر بناء CUDA الي كولاب
جايبه أصلاً. اعتمادهما الحقيقي (skimage، pandas، fire) ننصّبه صراحةً.

In [ ]:
!pip -q install fastapi "uvicorn[standard]" python-multipart pyngrok
!pip -q install --no-deps torchxrayvision medmnist
!pip -q install -q scikit-image fire

import importlib, sklearn
for m in ["fastapi", "uvicorn", "torchxrayvision", "joblib", "sklearn", "transformers", "skimage"]:
    try:
        mod = importlib.import_module(m)
        print("%-16s %s" % (m, getattr(mod, "__version__", "?")))
    except Exception as e:
        print("%-16s ✗ %s: %s" % (m, type(e).__name__, e))

# ملفات .joblib العربية انحفظت بـsklearn 1.7.2. الفجوة الكبيرة بالإصدار تخلي فك التسلسل
# يرمي تحذيراً — وأحياناً يفشل. إذا طلع خطأ بتحميل ar، هذا أول شي تشوفه.
if not sklearn.__version__.startswith("1.7"):
    print("\n[!] sklearn هنا %s بينما الـjoblib انحفظت بـ1.7.2 — إذا فشل تحميل ar:"
          % sklearn.__version__)
    print("    !pip -q install 'scikit-learn==1.7.2'  ثم Runtime ← Restart session")

## ٥) تشغيل الخادم

`main.py` يحمّل **كل** نموذج وقت الاستيراد — ١٤ نموذج صور + MARBERT + رؤوس عربية.
هذا ياخذ دقيقة إلى ثلاث، فالخلية تنطر `/health` بدل ما تفترض إنه قام، وتطبع ذيل السجل
إذا فشل بدل ما تعطيك خطأ اتصال أعمى.

In [ ]:
import os, sys, time, subprocess, requests

API_DIR = "/content/hub/api"
LOG = "/content/server.log"
PORT = 8000

# اقتل أي خادم من تشغيل سابق للخلية — وإلا الثاني يفشل بـ"address already in use"
# والسجل يمتلئ بخطأ يخص المنفذ مو النماذج.
subprocess.run(["pkill", "-f", "uvicorn main:app"], capture_output=True)
time.sleep(2)

proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd=API_DIR, stdout=open(LOG, "wb"), stderr=subprocess.STDOUT,
    env=dict(os.environ, PYTHONUNBUFFERED="1"))
print("uvicorn pid =", proc.pid)

DEADLINE = time.time() + 900          # ١٥ دقيقة: تحميل ١.٤ غيغا على معالج بارد
health = None
while time.time() < DEADLINE:
    if proc.poll() is not None:
        print("\n[✗] العملية ماتت. آخر ٤٠ سطر من السجل:\n")
        print(open(LOG, encoding="utf-8", errors="replace").read()[-4000:])
        raise SystemExit("الخادم ما قام — شوف السجل فوق.")
    try:
        r = requests.get("http://127.0.0.1:%d/health" % PORT, timeout=3)
        if r.ok:
            health = r.json()
            break
    except Exception:
        pass
    tail = open(LOG, encoding="utf-8", errors="replace").read().strip().splitlines()
    print("  %-78s" % (tail[-1][:78] if tail else "...قيد الإقلاع"), end="\r", flush=True)
    time.sleep(3)

if health is None:
    print(open(LOG, encoding="utf-8", errors="replace").read()[-4000:])
    raise SystemExit("انتهى وقت الانتظار.")

print("\n[✓] الخادم شغّال على", health["device"])
up = [k for k, v in health["models"].items() if v]
down = [k for k, v in health["models"].items() if not v]
print("    جاهز (%d): %s" % (len(up), ", ".join(up)))
if down:
    print("    غير متاح (%d): %s   <- checkpoint ناقص بالحزمة" % (len(down), ", ".join(down)))

## ٦) فتح النفق

In [ ]:
from pyngrok import ngrok, conf

if not NGROK_TOKEN:
    raise SystemExit("ماكو توكن ngrok — ارجع للخلية ٢.")

conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()                       # نفق قديم من نفس الجلسة يحجز الحصة المجانية
tunnel = ngrok.connect(PORT, "http")
URL = tunnel.public_url.replace("http://", "https://")

print("=" * 62)
print("  ", URL)
print("=" * 62)
print("""
افتح الرابط بالمتصفح — الصفحة تنخدم من نفس العنوان، وحقل «عنوان الخادم API»
يضبط نفسه تلقائياً على أصل الصفحة، فما تحتاج تلصق ولا شي.

أول فتح يعرض صفحة ngrok الاعتراضية: اضغط «Visit Site» مرة وحدة.

الرابط عام طول ما هذي الجلسة شغّالة. أطفيه بالخلية الأخيرة لمّا تخلص.
""")

## ٧) فحص سريع

يتأكد إن الرابط يشتغل **من برّا** الجلسة، مو بس على 127.0.0.1 — وهذا الفرق بين
«الخادم قام» و«النفق يوصله».

In [ ]:
import requests
h = {"ngrok-skip-browser-warning": "1"}   # تتخطى الصفحة الاعتراضية للطلبات البرمجية
try:
    r = requests.get(URL + "/health", headers=h, timeout=30)
    print("GET /health ->", r.status_code, r.json()["status"], "| device:", r.json()["device"])
    m = requests.get(URL + "/models", headers=h, timeout=60).json()
    print("GET /models ->", len(m["models"]), "نموذج")
    for x in m["models"][:6]:
        acc = (x.get("metrics") or {}).get("test_accuracy")
        print("   %-12s %s" % (x["id"], ("acc=%.4f" % acc) if acc else "—"))
except Exception as e:
    print("[✗] %s: %s" % (type(e).__name__, e))
    print("    إذا رجع 402/429 فهذي حصة ngrok المجانية، مو الخادم.")

## ٨) الإيقاف

شغّلها لمّا تخلص. الرابط يموت فوراً، والمنفذ ينفتح لتشغيلة جديدة.

In [ ]:
from pyngrok import ngrok
import subprocess
ngrok.kill()
subprocess.run(["pkill", "-f", "uvicorn main:app"], capture_output=True)
print("انطفى النفق والخادم.")

## ٩) ذيل السجل (للتشخيص)

شغّلها بأي وقت تشوف شنو يسوي الخادم — تحميل النماذج، الطلبات الواصلة، وأي استثناء.

In [ ]:
print(open("/content/server.log", encoding="utf-8", errors="replace").read()[-6000:])